<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring

This notebook audits the Week-5 model using a before/after validation comparison, a leakage audit, real error examples, and safer claim language.


In [2]:
!pip -q install duckdb scikit-learn pandas
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{BASE}/fact_content_daily_performance/**/*.parquet"
print("Connected to FlyRank Internship Warehouse")


Connected to FlyRank Internship Warehouse


# 1. Two paper findings + my methodology questions

**Important:** The assigned FlyRank research paper was not provided with this request, so I do not invent paper findings.

Before submission, replace the two placeholders below with **two exact findings from the assigned research paper**.

### Finding 1
**Paper finding:** [Replace with an exact finding from the FlyRank research paper.]

**My methodology question:** Where does the label or outcome for this finding come from? I would check whether it is directly observed, derived from a proxy, or based on another measurement process. I would also ask whether the validation design is strong enough for the level of the claim.

### Finding 2
**Paper finding:** [Replace with an exact finding from the FlyRank research paper.]

**My methodology question:** Does the validation design match the claim? For example, I would check whether evaluation is grouped, time-aware, or otherwise separated enough to avoid related examples appearing in both training and evaluation.

This is a constructive audit. The purpose is to understand how methodology supports claims, not to grade the paper.


In [3]:
paper_findings_check = {
    "finding_1_added": False,
    "finding_2_added": False
}
print("Before submission, replace both placeholders with exact findings from the assigned paper.")
print(paper_findings_check)


Before submission, replace both placeholders with exact findings from the assigned paper.
{'finding_1_added': False, 'finding_2_added': False}


# 2. My model under an honest split (before/after)

I compare:

- **Before:** a random row split.
- **After:** a grouped split by pseudonymized client.

March signals are features. April sessions create the later outcome. April information is never used as a feature.


In [4]:
query = f'''
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(ga4_sessions) AS sessions,
           SUM(ga4_engaged_sessions) AS engaged_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND ga4_sessions > 0
    GROUP BY 1, 2
),
april AS (
    SELECT client_hash_id, content_hash_id,
           SUM(ga4_sessions) AS april_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-04'
      AND ga4_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.impressions, m.clicks,
       m.sessions, m.engaged_sessions,
       COALESCE(a.april_sessions, 0) AS april_sessions
FROM march m
LEFT JOIN april a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id
'''
df = con.sql(query).df()
df["engagement_rate"] = (df["engaged_sessions"] / df["sessions"]).clip(0,1)
df["target_decline"] = (df["april_sessions"] < df["sessions"]).astype(int)

features = ["impressions","clicks","sessions","engaged_sessions","engagement_rate"]
X = df[features].copy()
y = df["target_decline"].copy()
groups = df["client_hash_id"].copy()
print("Rows:", len(df))
print("Positive decline rate:", round(y.mean(),3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 63650
Positive decline rate: 0.524


### Before: random row split

This split may be optimistic because related client patterns can appear in both training and test data.


In [5]:
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
random_model = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)
random_model.fit(Xtr_r, ytr_r)
random_auc = roc_auc_score(yte_r, random_model.predict_proba(Xte_r)[:,1])
print("Random row split ROC-AUC:", round(random_auc,3))


Random row split ROC-AUC: 0.787


### After: grouped-by-client split

This is the preferred validation design because each client appears only in training or testing.


In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

grouped_model = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)
grouped_model.fit(X_train, y_train)
grouped_prob = grouped_model.predict_proba(X_test)[:,1]
grouped_auc = roc_auc_score(y_test, grouped_prob)

print("Grouped split ROC-AUC:", round(grouped_auc,3))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Train positive rate:", round(y_train.mean(),3))
print("Test positive rate:", round(y_test.mean(),3))


Grouped split ROC-AUC: 0.793
Client overlap: 0
Train positive rate: 0.478
Test positive rate: 0.712


### Before/after comparison

The grouped result is treated as the more honest estimate because it prevents client overlap.


In [7]:
comparison = pd.DataFrame({
    "Validation design":["Random row split (before)","Grouped by client (after)"],
    "ROC-AUC":[random_auc, grouped_auc]
})
comparison["ROC-AUC"] = comparison["ROC-AUC"].round(3)
print(comparison.to_string(index=False))
print("\nDifference (random - grouped):", round(random_auc-grouped_auc,3))
if random_auc > grouped_auc:
    print("Observation: the random split produced a higher measured score, so row-level validation may be more optimistic.")
else:
    print("Observation: the grouped split was not lower on this run. The grouped design is still preferred because it prevents client overlap.")


        Validation design  ROC-AUC
Random row split (before)    0.787
Grouped by client (after)    0.793

Difference (random - grouped): -0.006
Observation: the grouped split was not lower on this run. The grouped design is still preferred because it prevents client overlap.


# 3. Leakage audit

The final feature set is checked again.

A feature is risky if it contains future information, the future outcome, a transformation of the future outcome, or information unavailable at prediction time.

All model features come from March. April sessions are used only to create the target.


In [8]:
leakage_audit = pd.DataFrame({
"Column":["impressions","clicks","sessions","engaged_sessions","engagement_rate","april_sessions","target_decline"],
"Available at March prediction time":["Yes","Yes","Yes","Yes","Yes","No","No"],
"Used as model feature":["Yes","Yes","Yes","Yes","Yes","No","No"],
"Leakage assessment":[
"Low risk: observed March signal",
"Low risk: observed March signal",
"Low risk: observed March signal",
"Low risk: observed March signal",
"Low risk: derived only from March signals",
"Leakage if used: future outcome information",
"Leakage if used: target itself"
]})
print(leakage_audit.to_string(index=False))
assert "april_sessions" not in features
assert "target_decline" not in features
print("\nLeakage check passed.")


          Column Available at March prediction time Used as model feature                          Leakage assessment
     impressions                                Yes                   Yes             Low risk: observed March signal
          clicks                                Yes                   Yes             Low risk: observed March signal
        sessions                                Yes                   Yes             Low risk: observed March signal
engaged_sessions                                Yes                   Yes             Low risk: observed March signal
 engagement_rate                                Yes                   Yes   Low risk: derived only from March signals
  april_sessions                                 No                    No Leakage if used: future outcome information
  target_decline                                 No                    No              Leakage if used: target itself

Leakage check passed.


### Real failure examples

- **False positive:** predicted decline, but decline did not occur.
- **False negative:** decline occurred, but the model did not predict it.

The examples below use pseudonymized data only.


In [9]:
error_df = df.iloc[test_idx].copy()
error_df["predicted_probability"] = grouped_prob
error_df["predicted_decline"] = (error_df["predicted_probability"] >= 0.5).astype(int)

false_positives = error_df[(error_df["predicted_decline"]==1)&(error_df["target_decline"]==0)]
false_negatives = error_df[(error_df["predicted_decline"]==0)&(error_df["target_decline"]==1)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives[features+["april_sessions","target_decline","predicted_probability"]].head(5))
print("\nExample false negatives:")
display(false_negatives[features+["april_sessions","target_decline","predicted_probability"]].head(5))


False positives: 519
False negatives: 3629

Example false positives:


,impressions,clicks,sessions,engaged_sessions,engagement_rate,april_sessions,target_decline,predicted_probability
13,227.0,7.0,12.0,1.0,0.083333,13.0,0,0.535486
267,399.0,9.0,21.0,3.0,0.142857,21.0,0,0.655451
270,22.0,2.0,6.0,0.0,0.000000,12.0,0,0.613854
298,9.0,0.0,3.0,0.0,0.000000,3.0,0,0.595444
311,47.0,6.0,5.0,0.0,0.000000,8.0,0,0.539190



Example false negatives:


,impressions,clicks,sessions,engaged_sessions,engagement_rate,april_sessions,target_decline,predicted_probability
259,84.0,3.0,3.0,0.0,0.0,1.0,1,0.424394
262,441.0,3.0,9.0,0.0,0.0,6.0,1,0.409990
263,3170.0,10.0,16.0,0.0,0.0,6.0,1,0.356461
265,33.0,0.0,3.0,0.0,0.0,2.0,1,0.489364
269,1124.0,9.0,12.0,0.0,0.0,4.0,1,0.329804


### Feature interpretation

Permutation importance shows how much measured ROC-AUC changes when a feature is shuffled. It is an interpretation tool, not evidence of causation.


In [10]:
importance = permutation_importance(
    grouped_model, X_test, y_test,
    scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1
)
importance_df = pd.DataFrame({
    "feature":features,
    "importance_mean":importance.importances_mean
}).sort_values("importance_mean", ascending=False)
importance_df["importance_mean"] = importance_df["importance_mean"].round(4)
print(importance_df.to_string(index=False))


         feature  importance_mean
        sessions           0.3305
     impressions           0.0315
          clicks           0.0296
engaged_sessions           0.0152
 engagement_rate           0.0074


# 4. Claim rewrite

### Previous bold claim

> "The Random Forest model predicts which content will decline and should be refreshed."

This wording goes further than the evidence because the model does not test whether refreshing content causes recovery.

### Rewritten safe claim

> "On the held-out grouped-by-client split, the Random Forest model measured the ROC-AUC shown in this notebook for ranking content items associated with a later decline in sessions. The result is directional decision-support for prioritizing human review. It does not show that refreshing content will cause performance recovery."

The rewritten claim is narrower and better matched to the validation evidence.


In [11]:
top_feature = importance_df.iloc[0]["feature"]
print("Claim audit summary")
print(f"Random split ROC-AUC: {random_auc:.3f}")
print(f"Grouped split ROC-AUC: {grouped_auc:.3f}")
print(f"Strongest measured feature: {top_feature}")
print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")
print("\nSafe interpretation: The model provides directional decision-support for prioritizing human review.")
print("It does not prove that refreshing content causes performance recovery.")


Claim audit summary
Random split ROC-AUC: 0.787
Grouped split ROC-AUC: 0.793
Strongest measured feature: sessions
False positives: 519
False negatives: 3629

Safe interpretation: The model provides directional decision-support for prioritizing human review.
It does not prove that refreshing content causes performance recovery.


# Self-check

- [ ] Replace the two paper-finding placeholders with exact findings from the assigned FlyRank research paper.
- [x] Random and grouped validation results are compared.
- [x] The grouped split checks for zero client overlap.
- [x] Future April sessions and the target are excluded from model features.
- [x] Real failure examples are included.
- [x] Claims use observed, measured, directional, and decision-support language.
- [ ] Run **Runtime → Run all** before committing.
- [ ] Confirm that no client names, URLs, or private queries appear.
- [ ] Commit as `work/notebooks/w06_validation_audit.ipynb`.

Do not submit invented paper findings.
